In [1]:
import tensorflow as tf; print('TensorFlow:', tf.__version__); print('GPUs:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.21.0
GPUs: []


## 1.2 Dataset audit

Scan every file for image metadata and a SHA-256 content hash. Display class counts, size statistics, unreadable files and exact duplicates below; no CSV files are written.


In [2]:
from pathlib import Path
import hashlib
import pandas as pd
from PIL import Image
from IPython.display import display

# The notebook runs from the project directory.
dataset_root = (Path.cwd() / "cashbowman" / "ai-generated-images-vs-real-images"
                / "versions" / "1")
class_folders = {"AI-generated": "AiArtData", "Real": "RealArt"}

def sha256_file(path):
    """Hash file contents in chunks, so large files do not fill memory."""
    digest = hashlib.sha256()
    with path.open("rb") as image_file:
        for chunk in iter(lambda: image_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def inspect_file(path, dataset_root, label):
    record = {
        "path": path.relative_to(dataset_root).as_posix(),
        "class": label,
        "file_size_bytes": "",
        "width": "",
        "height": "",
        "colour_mode": "",
        "format": "",
        "sha256": "",
        "unreadable": False,
        "error": "",
        "duplicate_of": "",
    }
    try:
        record["file_size_bytes"] = path.stat().st_size
        record["sha256"] = sha256_file(path)
        # verify() checks file integrity; load() also checks pixel decoding.
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            image.load()
            record.update(
                width=image.width,
                height=image.height,
                colour_mode=image.mode,
                format=image.format,
            )
    except Exception as error:
        # Keep the failed file in the audit instead of silently skipping it.
        record["unreadable"] = True
        record["error"] = f"{type(error).__name__}: {error}"
    return record

In [3]:
# Scan recursively because the class folder names are repeated inside the download.
records = []
first_seen = {}
for label, folder in class_folders.items():
    class_root = dataset_root / folder
    if not class_root.is_dir():
        raise FileNotFoundError(f"Missing dataset folder: {class_root}")
    paths = sorted(path for path in class_root.rglob("*") if path.is_file())
    if not paths:
        raise ValueError(f"No files found in {class_root}")
    for path in paths:
        row = inspect_file(path, dataset_root, label)
        content_hash = row["sha256"]
        if content_hash:
            if content_hash in first_seen:
                row["duplicate_of"] = first_seen[content_hash]
            else:
                first_seen[content_hash] = row["path"]
        records.append(row)

audit_df = pd.DataFrame(records)
for column in ["width", "height", "file_size_bytes"]:
    audit_df[column] = pd.to_numeric(audit_df[column], errors="coerce").astype("Int64")
print(f"Audited {len(audit_df)} files. The full per-image table is stored in audit_df.")
display(audit_df)


Audited 975 files. The full per-image table is stored in audit_df.


,path,class,file_size_bytes,width,height,colour_mode,format,sha256,unreadable,error,duplicate_of
0,AiArtData/AiArtData/-how-important-people-from...,AI-generated,97136,880,440,RGB,JPEG,410f52ab558884eba3b6b271b4f5ade076f6dc06adbf14...,False,,
1,AiArtData/AiArtData/-s-fluffy-fur-and-round-fe...,AI-generated,329446,2940,1960,RGB,JPEG,e31692a1b6ac7ff0754ea8717ad51e1cbed6267982bace...,False,,
2,AiArtData/AiArtData/-using-generative-ai-techn...,AI-generated,267045,1067,1690,RGB,JPEG,e90cbc6e6199b9f622295d61ef0913f2f64f0544c2f650...,False,,
3,AiArtData/AiArtData/-yellow-kimono-with-a-gold...,AI-generated,89671,800,800,RGB,JPEG,ea4b6260faac11d4914e56c123d87b97c692ab2c3802fa...,False,,
4,AiArtData/AiArtData/.amazonaws.com2Fpublic2Fim...,AI-generated,52677,600,480,RGB,JPEG,3009ba845a440f74016286b0603c244c681a93e1f4a14f...,False,,
...,...,...,...,...,...,...,...,...,...,...,...
970,RealArt/RealArt/Women-clothing-Kenya-East-Afri...,Real,267179,1600,836,RGB,JPEG,e1e3f9dca6ace7a72f6ade3e73c9c76c7adde58a11707e...,False,,
971,RealArt/RealArt/Worlds-15-Most-Iconic-Portrait...,Real,70700,552,800,RGB,JPEG,893f967dc76df6678f85027dc636941ffb41dae9bdd0d1...,False,,
972,RealArt/RealArt/WR202206_GiraffeTranslocation_...,Real,390161,1200,630,RGB,JPEG,99e573355842fdd6919b832593f7e9429ca98ec13bc313...,False,,
973,RealArt/RealArt/XaHpf_z51huQS_JPHs-jkPhBp0dLlx...,Real,636250,2048,1365,RGB,JPEG,c1ee99f18fe1b880896cacaaf1e15e01085aec08fe1b06...,False,,


In [4]:
# Counts and quality checks for each class.
counts = audit_df.groupby("class").agg(
    files=("path", "size"),
    readable_images=("unreadable", lambda values: (~values).sum()),
    corrupt_or_unreadable=("unreadable", "sum"),
    duplicate_files=("duplicate_of", lambda values: values.ne("").sum()),
)
print("Counts per class (duplicates count additional copies beyond the first):")
display(counts)

# Missing dimensions from unreadable files are excluded automatically.
# File-size statistics include every file whose size could be read.
size_stats = audit_df.groupby("class")[["width", "height", "file_size_bytes"]].agg(
    ["min", "median", "max"]
)
print("Image dimensions in pixels and file sizes in bytes:")
display(size_stats)

unreadable_files = audit_df.loc[audit_df["unreadable"], ["path", "class", "error"]]
print(f"Corrupt or unreadable files: {len(unreadable_files)}")
display(unreadable_files)

# Show both the original and every extra copy in each duplicate group.
duplicate_images = audit_df.loc[
    audit_df["sha256"].ne("") & audit_df["sha256"].duplicated(keep=False),
    ["path", "class", "sha256", "duplicate_of"],
].sort_values("sha256")
print(f"Exact duplicate groups: {duplicate_images['sha256'].nunique()}")
print(f"Additional duplicate files: {audit_df['duplicate_of'].ne('').sum()}")
with pd.option_context("display.max_colwidth", None):
    display(duplicate_images)
print("SHA-256 detects identical file bytes; visually similar images may have different hashes.")


Counts per class (duplicates count additional copies beyond the first):


,files,readable_images,corrupt_or_unreadable,duplicate_files
class,,,,
AI-generated,539,539,0,1
Real,436,436,0,0


Image dimensions in pixels and file sizes in bytes:


width               height              file_size_bytes  \
               min  median   max    min median   max             min   
class                                                                  
AI-generated   225  1024.0  6000    135  800.0  4001           11738   
Real           183   920.0  8495    158  720.0  7091           11311   

                                  
                median       max  
class                             
AI-generated  199760.0   5626183  
Real          144840.0  10195352

Corrupt or unreadable files: 0


,path,class,error


Exact duplicate groups: 1
Additional duplicate files: 1


,path,class,sha256,duplicate_of
346,AiArtData/AiArtData/image33.jpg,AI-generated,f3268a4fc0477b2f3d9728f118afe32bf5196c3b476def53e34229672f77dc97,
347,AiArtData/AiArtData/image34.jpg,AI-generated,f3268a4fc0477b2f3d9728f118afe32bf5196c3b476def53e34229672f77dc97,AiArtData/AiArtData/image33.jpg


SHA-256 detects identical file bytes; visually similar images may have different hashes.
